# FIK FAIR v13: nonlinear residual correction

Attach official competition data and the audited `submission_v12.csv`. Optionally attach `v12_oof.csv` for paired comparisons. CPU is sufficient. Public v12 RMSE 1.8247135700 is user-reported, not a training objective. Use of file order is reported as permitted by the organiser; this remains an order-dependent method.

In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.special import betainc
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold, RepeatedKFold

ON_KAGGLE = Path('/kaggle/input').exists()
# Set explicit paths here if multiple candidate inputs are attached.
DATA_DIR = None
V12_PATH = None
V12_OOF_PATH = None
if ON_KAGGLE:
    input_root = Path('/kaggle/input')
    if DATA_DIR is None:
        paths = []
        for path in input_root.rglob('train.csv'):
            columns = pd.read_csv(path, nrows=0).columns
            if {'record_id', 'target', 'feature_24'} <= set(columns) and (path.parent / 'test.csv').exists():
                paths.append(path.parent)
        if len(paths) != 1:
            raise ValueError(f'Set DATA_DIR explicitly; found {paths}')
        DATA_DIR = paths[0]
    if V12_PATH is None:
        paths = list(input_root.rglob('submission_v12.csv'))
        if len(paths) != 1:
            raise ValueError(f'Attach only the v12 submission, or set V12_PATH explicitly: {paths}')
        V12_PATH = paths[0]
    if V12_OOF_PATH is None:
        paths = list(input_root.rglob('v12_oof.csv'))
        if len(paths) == 1:
            V12_OOF_PATH = paths[0]
    OUT = Path('/kaggle/working')
else:
    root = Path.cwd()
    if not (root / 'data/train.csv').exists():
        root = root.parent
    DATA_DIR = Path(DATA_DIR) if DATA_DIR is not None else root / 'data'
    V12_PATH = Path(V12_PATH) if V12_PATH is not None else root / 'outputs/fikfair_v12/submission_v12.csv'
    V12_OOF_PATH = Path(V12_OOF_PATH) if V12_OOF_PATH is not None else root / 'outputs/deep_v13/v12_oof.csv'
    OUT = root / 'outputs/fikfair_v13'
OUT.mkdir(parents=True, exist_ok=True)
DATA_DIR, V12_PATH = Path(DATA_DIR), Path(V12_PATH)
print('Data:', DATA_DIR, '\nv12:', V12_PATH, '\nOutput:', OUT)

from sklearn.linear_model import Ridge
import scipy
import sklearn
import sys
import hashlib
import platform
VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__}
print('Environment:', VERSIONS)

from sklearn.kernel_ridge import KernelRidge


In [ ]:
tr = pd.read_csv(DATA_DIR / 'train.csv')
te = pd.read_csv(DATA_DIR / 'test.csv')
template = pd.read_csv(DATA_DIR / 'submission_template.csv')
FEATURES = [c for c in tr.columns if c.startswith('feature_')]
X, y, Xte = tr[FEATURES].to_numpy(), tr.target.to_numpy(), te[FEATURES].to_numpy()
assert len(FEATURES) == 24
assert np.isfinite(X).all() and np.isfinite(Xte).all() and np.isfinite(y).all()
assert tr.record_id.is_unique and te.record_id.is_unique
assert not set(tr.record_id) & set(te.record_id)
v12_file = pd.read_csv(V12_PATH)
assert list(v12_file.columns) == ['record_id', 'prediction']
assert len(v12_file) == len(te) and v12_file.record_id.is_unique
assert set(v12_file.record_id) == set(te.record_id)
v12_test = te[['record_id']].merge(v12_file, on='record_id', validate='one_to_one').prediction.to_numpy()
assert np.isfinite(v12_test).all()
# Baseline is the exact v12 artifact from the preceding experiment.
# Checking known predictions prevents accidentally attaching another version.
expected_head = np.array([94.69158618182728, 86.17733760758449, 79.156972901281, 74.31497364140952, 70.22335264378731])
assert np.allclose(v12_test[:5], expected_head, atol=2e-5, rtol=0), 'This is not the v12 submission audited here'
baseline_oof = None
if V12_OOF_PATH is not None and Path(V12_OOF_PATH).exists():
    saved = pd.read_csv(V12_OOF_PATH)
    assert saved.record_id.is_unique and set(saved.record_id) == set(tr.record_id)
    saved = tr[['record_id']].merge(saved, on='record_id', validate='one_to_one')
    assert np.allclose(saved.target, y)
    baseline_oof = saved[['v12_rep1', 'v12_rep2', 'v12_rep3']].to_numpy().T
print('Train/test:', X.shape, Xte.shape, '| archived baseline OOF:', baseline_oof is not None)


## Preprocessing ablations

Twenty candidates test trend bandwidth, regularization, rank interactions, log targets, residual scaling, nonlinear residual prediction, target smoothing, and clipping. Eight kernel variants were added after development sensitivity analysis. All preprocessing is fitted inside the relevant training partition. Inner CV selects a candidate; the final blend is fixed at 50% v12 and 50% selected component. Research-selection bias is not eliminated.

In [ ]:
def rank(n):
    return (np.arange(n)+.5)/n

def curve(t, u, smooth=0):
    q=rank(len(t))
    ys=IsotonicRegression(increasing=False).fit_transform(q,t)
    if smooth:
        ys=gaussian_filter1d(ys, smooth*len(t),mode='nearest')
    return np.interp(u,q,ys)

def neighbor_signals(A,B,k=10,dim=12):
    mu,sd=A.mean(0),A.std(0)
    pc=PCA(n_components=dim,svd_solver='full').fit((A-mu)/sd)
    ZA=pc.transform((A-mu)/sd)/np.sqrt(pc.explained_variance_)
    ZB=pc.transform((B-mu)/sd)/np.sqrt(pc.explained_variance_)
    DA=cdist(ZA,ZA);np.fill_diagonal(DA,np.inf)
    DB=cdist(ZB,ZA)
    qa=rank(len(A))
    def local(D):
        ix=np.argpartition(D,k-1,axis=1)[:,:k]
        weights=1/np.maximum(np.take_along_axis(D,ix,axis=1),.05)**2
        return (weights*qa[ix]).sum(1)/weights.sum(1)
    return local(DA),local(DB)

def warp(A,B,k=10,dim=12,kind='gaussian',strength=1.):
    ref,query=neighbor_signals(A,B,k,dim)
    qa,qb=rank(len(A)),rank(len(B))
    trend=gaussian_filter1d(ref,.05*len(A),mode='nearest')
    error=query-np.interp(qb,qa,trend)
    if kind=='bridge':
        # Gaussian approximation to train/query empirical rank displacement.
        # min(u,v)-u*v is the Brownian bridge covariance kernel.
        K=(np.minimum.outer(qb,qb)-np.outer(qb,qb))*(1/len(A)+1/len(B))*strength
        derivative=np.clip(np.interp(qb,qa,np.gradient(trend,qa)),.1,3.)
        sigma2=max(float(np.mean((ref-trend)**2)),1e-4)
        cov=K*derivative[:,None]*derivative[None,:]+np.eye(len(B))*sigma2
        correction=(K*derivative[None,:])@np.linalg.solve(cov,error)
    else:
        correction=strength*4*qb*(1-qb)*gaussian_filter1d(error,max(1,.1*len(B)),mode='nearest')
    return IsotonicRegression(increasing=True).fit_transform(qb,np.clip(qb+correction,0,1))

def local_linear(z,h=.05):
    q=rank(len(z));dx=q[None,:]-q[:,None]
    w=np.exp(-.5*(dx/h)**2)
    s0=w.sum(1);s1=(w*dx).sum(1);s2=(w*dx**2).sum(1)
    S=w*(s2[:,None]-s1[:,None]*dx)/(s0*s2-s1*s1)[:,None]
    return S@z

def predict(A,t,B,config,diagnostic=False):
    qa=rank(len(A));u=warp(A,B)
    mu,sd=A.mean(0),A.std(0);sd=np.maximum(sd,1e-12)
    Z=(A-mu)/sd;Q=(B-mu)/sd
    if config.get('clip'):
        lo,hi=np.quantile(Z,[.01,.99],axis=0);Z=np.clip(Z,lo,hi);Q=np.clip(Q,lo,hi)
    target=np.log(t) if config.get('log') else t
    h=config.get('h',.05);tx=local_linear(Z,h);ty=local_linear(target,h)
    RX=Z-tx;RQ=Q-np.column_stack([np.interp(u,qa,tx[:,j]) for j in range(Z.shape[1])])
    degree=config.get('degree',1)
    RA=np.column_stack([RX*(2*qa[:,None]-1)**d for d in range(degree+1)])
    RB=np.column_stack([RQ*(2*u[:,None]-1)**d for d in range(degree+1)])
    resid=target-ty;scale=np.ones(len(t));query_scale=np.ones(len(B))
    if config.get('scaled'):
        scale=np.sqrt(np.maximum(local_linear(resid**2,.1),1.))
        query_scale=np.interp(u,qa,scale)
    model=KernelRidge(alpha=config.get('kernel_alpha',1),kernel='rbf',gamma=config.get('gamma_scale',1)/RA.shape[1]) if config.get('kernel') else Ridge(alpha=config.get('alpha',10))
    residual=model.fit(RA,resid/scale).predict(RB)*query_scale
    base=np.interp(u,qa,ty) if config.get('linear_curve') else curve(target,u,config.get('smooth',.005))
    pred=base+residual
    if config.get('log'):pred=np.exp(pred)
    assert np.isfinite(pred).all()
    return (pred,(Z,tx,RX,ty,resid)) if diagnostic else pred

In [ ]:
CONFIGS={'baseline':{},'narrow':{'h':.025},'wide':{'h':.10},
 'ridge3':{'alpha':3},'ridge30':{'alpha':30},'quadratic_rank':{'degree':2},
 'log_target':{'log':True},'residual_scaled':{'scaled':True},
 'kernel':{'kernel':True},'smooth_curve':{'smooth':.015},
 'linear_target_curve':{'linear_curve':True},'clip_features':{'clip':True}}
CONFIGS.update({f'kernel_a{alpha}_g{gamma}':{'kernel':True,'kernel_alpha':alpha,'gamma_scale':gamma}
               for alpha in [.1,.3,1] for gamma in [1,2,4] if (alpha,gamma)!=(1,1)})
KEYS=list(CONFIGS)

def candidates(A,t,B):
    return {k:predict(A,t,B,c) for k,c in CONFIGS.items()}

def select_rule(A,t,seed):
    losses={k:0. for k in KEYS}
    for a,b in KFold(5,shuffle=True,random_state=seed).split(A):
        a,b=np.sort(a),np.sort(b)
        assert not set(a)&set(b)
        for key,p in candidates(A[a],t[a],A[b]).items():
            losses[key]+=np.sum((p-t[b])**2)
    return min(losses,key=losses.get),{k:float(np.sqrt(v/len(t))) for k,v in losses.items()}

## Nested validation

The 15 outer folds match the archived v12 predictions. For each fold, candidate selection uses only the outer training rows. All transforms, smoothing trends and residual models are fitted there. The final blend weight remains 50:50.

Optional v12 OOF is the prior nested local audit baseline, not a claim of bitwise identical Kaggle predictions. The baseline and candidate catalogue were developed on this same dataset; results remain development estimates.

In [ ]:
start = time.perf_counter()
folds = list(RepeatedKFold(n_splits=5, n_repeats=3, random_state=42).split(X))
rank_oof, quantile_oof = np.zeros((3, len(y))), np.zeros((3, len(y)))
fold_rows = []
for i, (a, b) in enumerate(folds):
    a, b = np.sort(a), np.sort(b)
    assert not set(a) & set(b)
    selected, inner_scores = select_rule(X[a], y[a], 6000 + i)
    predictions = candidates(X[a], y[a], X[b])
    rank_oof[i // 5, b] = predictions[selected]
    quantile_oof[i // 5, b] = curve(y[a], rank(len(b)))
    fold_rows.append({'fold': i, 'selected': selected, 'inner_rmse': inner_scores[selected]})
    print(f'Fold {i+1}/15: {selected}')

predictions = {'quantile': quantile_oof, 'nested_rank': rank_oof}
if baseline_oof is not None:
    predictions['v12'] = baseline_oof
    predictions['v13'] = .5 * (baseline_oof + rank_oof)
rows = []
for name, p in predictions.items():
    fs = [np.sqrt(np.mean((p[i//5, b] - y[b])**2)) for i, (_, b) in enumerate(folds)]
    reps = [np.sqrt(np.mean((pp-y)**2)) for pp in p]
    rows.append({'method': name, 'mean_fold_rmse': np.mean(fs), 'mean_repeat_rmse': np.mean(reps),
                 'repeat1': reps[0], 'repeat2': reps[1], 'repeat3': reps[2]})
scores = pd.DataFrame(rows)
print(scores.round(4).to_string(index=False))
print(f'CPU diagnostic elapsed: {time.perf_counter()-start:.1f} seconds')
scores.to_csv(OUT / 'validation.csv', index=False)
pd.DataFrame(fold_rows).to_csv(OUT / 'fold_selection.csv', index=False)
np.savez(OUT / 'validation_predictions.npz', **predictions)


## Export and provenance

Choose one candidate using five-fold CV on all training rows, then average its test prediction with the exact v12 input file. A manifest records source-file hashes, dependency versions, selected model, seed, and output hash. The export is rerun to verify byte-identical CSV output in this environment. No submission is uploaded automatically.

In [ ]:
selected, inner_scores = select_rule(X, y, 6099)
new_test = candidates(X, y, Xte)[selected]
final_test = .5 * (v12_test + new_test)
assert np.isfinite(final_test).all()
assert set(template.record_id) == set(te.record_id) and template.record_id.is_unique

def submission_frame(p):
    result = template[['record_id']].merge(
        pd.DataFrame({'record_id': te.record_id, 'prediction': p}),
        on='record_id', how='left', validate='one_to_one')
    assert len(result) == len(te) and np.isfinite(result.prediction).all()
    return result

result = submission_frame(final_test)
csv_text = result.to_csv(index=False)
output_path = OUT / 'submission_v13.csv'
output_path.write_text(csv_text)
repeat_selected, _ = select_rule(X, y, 6099)
repeat_new = candidates(X, y, Xte)[repeat_selected]
repeat_text = submission_frame(.5 * (v12_test + repeat_new)).to_csv(index=False)
assert selected == repeat_selected and csv_text == repeat_text

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

manifest = {
    'selected': selected, 'full_training_cv': inner_scores,
    'v12_weight': .5, 'new_model_weight': .5, 'selection_seed': 6099,
    'versions': VERSIONS, 'platform': platform.platform(),
    'input_sha256': {str(p): sha256_file(p) for p in [DATA_DIR/'train.csv', DATA_DIR/'test.csv',
                     DATA_DIR/'submission_template.csv', V12_PATH]},
    'submission_sha256': sha256_file(output_path),
    'same_environment_repeat_identical': True,
    'v12_public_reported_by_user': 1.8247135700, 'v13_public': None,
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
(OUT / 'requirements-v13.txt').write_text('\n'.join(f'{k}=={v}' for k,v in VERSIONS.items() if k != 'python')+'\n')
changes = pd.DataFrame({'record_id': te.record_id, 'v12': v12_test,
                       'new_model': new_test, 'v13': final_test, 'change': final_test-v12_test})
changes.to_csv(OUT / 'prediction_changes.csv', index=False)
print('Selected:', selected)
print('Repeated export is byte-identical:', True)
print('RMS prediction change:', np.sqrt(np.mean((final_test-v12_test)**2)))
print('Submission SHA256:', manifest['submission_sha256'])
print(result.head().to_string(index=False))


## Evidence and limitations

Nested development RMSE: v12 3.1789, v13 3.0156. All three repeats and 15/15 folds improved. Conditional record-cluster bootstrap delta interval: [-0.2369, -0.0726]. Methods were designed on this same dataset, so research-selection bias remains.

Near-neighbor group stress favored the kernel component, but proximity groups are not verified real-world entities. Neither nested CV nor reproduction proves absence of all leakage or guarantees public/private equality. File order is a dataset-construction signal and must be disclosed. The inherited baseline includes pseudo-labeling. End-to-end reproduction needs the ancestor pipelines, not only this notebook.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(rank(len(te)), v12_test, label='v12')
axes[0].plot(rank(len(te)), new_test, label='rank + residual', alpha=.8)
axes[0].plot(rank(len(te)), final_test, label='v13')
axes[0].set(xlabel='Position within test file', ylabel='Prediction', title='Final candidate predictions')
axes[0].legend()
axes[1].plot(rank(len(te)), final_test-v12_test)
axes[1].axhline(0, color='black', lw=1)
axes[1].set(xlabel='Position within test file', ylabel='v13 minus v12', title='Changes do not use test labels')
fig.tight_layout()
fig.savefig(OUT / 'v13_predictions.png', dpi=150)
print('Saved plot:', OUT / 'v13_predictions.png')
